In [2]:
import pandas as pd
import re
from itables import show

data_raw = pd.read_parquet("data.parquet")


(2643425, 70)
material_id                                                          1__2
components                                        methane, carbon dioxide
thermoml_fair_version                                              1.0.25
property                                      Thermal conductivity, W/m/K
value                                                             0.02197
                                                         ...             
Lower temperature, K                                                  NaN
Upper temperature, K                                                  NaN
Amount concentration (molarity), mol/dm3_3                            NaN
Volume ratio of solute to solvent                                     NaN
formula                                                                  
Name: 100, Length: 70, dtype: object


Loading ITables v2.7.0 from the internet... (need help?)


In [3]:
print(data_raw.shape)
print(data_raw.iloc[1000])
show(data_raw.iloc[1000])

(2643425, 70)
material_id                                                  1__4
components                                     hexane, hexadecane
thermoml_fair_version                                      1.0.25
property                                      Mass density, kg/m3
value                                                       737.1
                                                     ...         
Lower temperature, K                                          NaN
Upper temperature, K                                          NaN
Amount concentration (molarity), mol/dm3_3                    NaN
Volume ratio of solute to solvent                             NaN
formula                                                          
Name: 1000, Length: 70, dtype: object


Loading ITables v2.7.0 from the internet... (need help?)


In [7]:
mix_counts = data_raw["components"].value_counts()
print(mix_counts.head(50))

components
germanium iron lithium oxide (Ge2FeLiO6), germanium iron sodium oxide (Ge2FeNaO6)    15941
decane                                                                               10684
pentaerythritol tetrahexanoate                                                       10246
pentaerythritol tetraheptanoate                                                       7686
ammonium nitrate, water, calcium nitrate                                              7444
1,1,1,2,3,3,3-heptafluoropropane                                                      7428
water                                                                                 6530
pentaerythritol tetrapentanoate                                                       5403
ethanol, water                                                                        5038
pentaerythritol tetrabutyrate                                                         4770
pentaerythritol tetraoctanoate                                                 

In [11]:
tmp = data_raw.copy()
tmp["comp_list"] = tmp["components"].astype(str).str.lower().str.split(r",\s+")
tmp["ncomp"] = tmp["comp_list"].apply(lambda xs: len(set(xs)))
tmp["mix_key"] = tmp["comp_list"].apply(lambda xs: "|".join(sorted(set(xs))))

binary = tmp[tmp["ncomp"] == 2]

summary = (
    binary.groupby("mix_key")["source_file"].nunique()
          .sort_values(ascending=False)
)

print("True binary mixtures (fixed parsing):", binary["mix_key"].nunique())
print(binary["mix_key"].value_counts().head(20))
     # top 30 by #files

True binary mixtures (fixed parsing): 28044
mix_key
germanium iron lithium oxide (ge2felio6)|germanium iron sodium oxide (ge2fenao6)    15941
ethanol|water                                                                        5121
carbon dioxide|methanol                                                              4546
triethylene glycol|water                                                             3638
methanol|water                                                                       3617
difluoromethane|pentafluoroethane                                                    3195
carbon dioxide|water                                                                 3081
2-aminoacetic acid|water                                                             3019
propan-2-ol|water                                                                    2967
propan-1-ol|water                                                                    2789
diisopropyl ether|propan-2-ol                   

In [13]:
binary["mix_key"].to_csv("Binary List 2")

In [ ]:
# This proves the new list is more accurate, and we checked how much they differ:
list1 = pd.read_csv("Binary List 1")
list2 = pd.read_csv("Binary List 2")
set1 = set(list1["mix_key_fixed"])
set2 = set(list2["mix_key"])
only_in_set1 = sorted(set1 - set2)
only_in_set2 = sorted(set2 - set1)

print("Only in set1:", len(only_in_set1))
print(only_in_set1[:50])  # show first 50
print("Only in set2:", len(only_in_set2))
print(only_in_set2[:50])

import pandas as pd
pd.Series(only_in_set1, name="only_in_set1").to_csv("only_in_set1.csv", index=False)
pd.Series(only_in_set2, name="only_in_set2").to_csv("only_in_set2.csv", index=False)


Only in set1: 174
['(e|e)-8,10-dodecadien-1-ol', '(r*|r*)-1,1,1,2,2,3,4,5,5,5-decafluoropentane', '(r*|s*)-1,1,1,2,2,3,4,5,5,5-decafluoropentane', '(z|z)-9,12-octadecadienoic acid', '.alpha.|.alpha.,4-trimethyl-3-cyclohexene-1-methanol', '.alpha.|.beta.-methyl-2-deoxy-d-ribofuranoside', '.alpha.|.beta.-trehalose monohydrate', ".beta.|.beta.'-dihydroxydiethyl sulfide", ".gamma.|.gamma.'-bipyridyl", ".kappa.o')bis(1,10-phenanthroline-.kappa.n1,.kappa.n10)diholmium|tetrakis[.mu.-(benzoato-.kappa.o:.kappa.o')]bis(benzoato-.kappa.o", '.psi.|.psi.-carotene', "2,2'-bipyridine-n|n'-dioxide", '2-(6-oxido-6h-dibenz[c|e][1,2]oxaphosphorin-6-yl)-1,4-dihydroxy phenylene', '2-(acetyloxy)-n-ethyl-n|n-dimethylethanaminium 1,1,1-trifluoro-n-[(trifluoromethyl)sulfonyl]methanesulfonamide', '2-(n|n-dimethylamino)ethanol', '2-diphenylmethoxy-n|n-dimethylethanamine hydrochloride', '2-hydroxy-n|n-dimethylethan-1-aminium acetate', '2-hydroxy-n|n-dimethylethan-1-aminium butyrate', '2-hydroxy-n|n-dimethylethan-

In [ ]:
print("Number of unique binary mixtures:", summary.shape[0])
print(summary.head(30))     